# 01  Carga, muestreo, traducción y limpieza del corpus

Proyecto 1: Análisis morfosintáctico de reseñas de cine con POS Tagging (NLTK y spaCy)

Este notebook cubre las etapas 1 y 2 del pipeline: deja listo el corpus que usan los demás notebooks.

Etapa

 1. Recolección y traducción
Carga del dataset, filtro por longitud, muestreo estratificado y traducción al español  
 2. Limpieza
Quitar ruido, duplicados y traducciones fallidas

Dataset: IMDB Spoiler Dataset (Kaggle, rmisra/imdb-spoiler-dataset), con dos archivos: reseñas y detalles de películas.
Traducción: modelo local Helsinki-NLP/opus-mt-en-es (MarianMT), en Google Colab con GPU.

 Orden importante: primero se filtra y se muestrea, y solo después se traduce la muestra (6.400 reseñas, no las 344.000 disponibles).

## 0. Configuración y decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Longitud de reseña | 30 a 400 palabras | Requisito del PDF (controla el tiempo de traducción y etiquetado) |
| Géneros | Drama, Comedy, Action, Horror, Animation | Los 5 géneros de las preguntas del eje A |
| Película con varios géneros | Se usa el primer género de la lista | Criterio consistente: cada reseña queda en un solo género, sin repetirse |
| Periodos | Antes de 2010 / 2010 en adelante, según la fecha de la reseña| Deja 3.200 reseñas en cada periodo |
| Cuota | 640 reseñas por celda género × periodo (10 celdas = 6.400) | Corpus de ~6.000 reseñas con margen para la limpieza |
| Tope por película | Dinámico: reparte la cuota entre las películas de cada celda lo más parejo posible | Evita que pocas películas dominen un género (Terror solo tiene 46) |
| Semilla | 42 | Reproducibilidad |
| Paquetes | 3 | Reparto de la traducción entre los 3 integrantes |
| Valoración | negativa 1-4, intermedia 5-6, positiva 7-10 | Regla del PDF |

In [1]:
#se instala kagglehub para descargar los conjuntos de datos
#!pip install -q kagglehub sentencepiece sacremoses

#importacion de librerías
import os, re, time
import numpy as np
import pandas as pd
import nltk
import torch
import kagglehub
from transformers import MarianMTModel, MarianTokenizer
import ssl
import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download("punkt")
nltk.download("punkt_tab")

#Parámetros del muestreo
SEED = 42
GENEROS = ["Drama", "Comedy", "Action", "Horror", "Animation"]
MIN_PALABRAS, MAX_PALABRAS = 30, 400
CUOTA_CELDA = 640
N_PAQUETES = 3
CORTE_PERIODO = 2010

#Carpeta de trabajo: Google Drive si estamos en Colab; si no, data/processed
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = "/content/drive/MyDrive/proyecto_pos"
except ImportError:
    CARPETA = "/data/processed"
os.makedirs(CARPETA, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)
print("GPU disponible:", torch.cuda.is_available())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Mounted at /content/drive
Carpeta de trabajo: /content/drive/MyDrive/proyecto_pos
GPU disponible: True


## 1. Carga y filtro

El archivo de reseñas pesa 900 MB, así que se lee por bloques y cada bloque se filtra antes de guardarlo. Se hace así:

1. De las películas se conserva el primer género y solo las de los 5 géneros elegidos.
2. De las reseñas se quitan los <br />, se normalizan los espacios y se filtra por longitud (30 a 400 palabras).
3. Se descartan reseñas sin fecha o sin calificación y las duplicadas.

In [2]:
#ruta del dataset
RUTA = kagglehub.dataset_download("rmisra/imdb-spoiler-dataset")
print(os.listdir(RUTA))

#Películas: primer género
peliculas = pd.read_json(f"{RUTA}/IMDB_movie_details.json", lines=True)
peliculas["genero"] = peliculas["genre"].str[0]
peliculas = peliculas[peliculas["genero"].isin(GENEROS)][["movie_id", "genero"]]
print("Películas en los 5 géneros:", len(peliculas))

#Reseñas: lectura por bloques con filtro
bloques = []
lector = pd.read_json(f"{RUTA}/IMDB_reviews.json", lines=True, chunksize=100_000, convert_dates=False)
for bloque in lector:
    bloque = bloque[bloque["movie_id"].isin(peliculas["movie_id"])].copy()
    bloque["review_text"] = (bloque["review_text"]
                             .str.replace(r"<br\s*/?>", " ", regex=True)
                             .str.replace(r"\s+", " ", regex=True).str.strip())
    n_palabras = bloque["review_text"].str.split().str.len()
    bloque = bloque[(n_palabras >= MIN_PALABRAS) & (n_palabras <= MAX_PALABRAS)]
    bloques.append(bloque[["movie_id", "review_date", "rating", "is_spoiler", "review_text"]])

rev = pd.concat(bloques, ignore_index=True)
rev["fecha"] = pd.to_datetime(rev["review_date"], errors="coerce")
rev["rating"] = pd.to_numeric(rev["rating"], errors="coerce")
rev = rev.dropna(subset=["fecha", "rating"]).drop_duplicates("review_text")
rev = rev.merge(peliculas, on="movie_id")

#Columnas de agrupación
rev["anio_review"] = rev["fecha"].dt.year
rev["periodo"] = np.where(rev["anio_review"] < CORTE_PERIODO, "antes_2010", "2010_en_adelante")
rev["valoracion"] = pd.cut(rev["rating"], [0, 4, 6, 10], labels=["negativa", "intermedia", "positiva"]).astype(str)

print("\nReseñas disponibles tras el filtro:", len(rev))
print("\nPor género:")
print(rev.groupby("genero").agg(resenas=("review_text", "size"), peliculas=("movie_id", "nunique")))
print("\nPor periodo:")
print(rev["periodo"].value_counts())

100%|██████████| 331M/331M [00:05<00:00, 63.4MB/s]

Extracting files...


['IMDB_movie_details.json', 'IMDB_reviews.json']
Películas en los 5 géneros: 1174

Reseñas disponibles tras el filtro: 344215

Por género:
           resenas  peliculas
genero                       
Action      149975        424
Animation    21829         84
Comedy       71806        345
Drama        83745        273
Horror       16860         46

Por periodo:
periodo
antes_2010          185141
2010_en_adelante    159074
Name: count, dtype: int64


## 2. Muestreo estratificado

Se arman 10 celdas (5 géneros × 2 periodos) de 640 reseñas cada una. Dentro de cada celda se limita cuántas reseñas puede aportar una misma película (el tope se calcula solo: es el menor valor con el que todavía se alcanza la cuota). Después cada celda se reparte en 3 paquetes parejos, para que cada integrante traduzca la misma mezcla de géneros y periodos.

In [3]:
def muestrear_celda(df, cuota):
    #Cuántas reseñas tiene cada película dentro de la celda
    disponibles = df.groupby("movie_id").size()
    #Menor tope por película con el que todavía se alcanza la cuota
    tope = 1
    while np.minimum(disponibles, tope).sum() < cuota:
        tope += 1
    df = df.sample(frac=1, random_state=SEED)               #mezclar
    df = df[df.groupby("movie_id").cumcount() < tope]       #máximo 'tope' por película
    return df.sample(cuota, random_state=SEED)

partes = []
for (genero, periodo), celda in rev.groupby(["genero", "periodo"]):
    print(f"{genero:10s} {periodo:17s} disponibles={len(celda):6d}  películas={celda['movie_id'].nunique()}")
    partes.append(muestrear_celda(celda, CUOTA_CELDA))

m = pd.concat(partes).sample(frac=1, random_state=SEED).reset_index(drop=True)
m["id"] = np.arange(1, len(m) + 1)
m["paquete"] = m.groupby(["genero", "periodo"]).cumcount() % N_PAQUETES + 1
m = m.rename(columns={"review_text": "texto_en", "rating": "calificacion"})
m = m[["id", "paquete", "movie_id", "genero", "periodo", "anio_review", "fecha",
       "calificacion", "valoracion", "is_spoiler", "texto_en"]]

print("\nTotal de la muestra:", len(m))
print("\nGénero x periodo:")
print(pd.crosstab(m["genero"], m["periodo"]))
print("\nPaquete x género:")
print(pd.crosstab(m["paquete"], m["genero"]))
print("\nValoración por género:")
print(pd.crosstab(m["genero"], m["valoracion"]))
print("\nMáximo de reseñas de una misma película, por género:")
print(m.groupby(["genero", "movie_id"]).size().groupby("genero").max())

Action     2010_en_adelante  disponibles= 78426  películas=424
Action     antes_2010        disponibles= 71549  películas=271
Animation  2010_en_adelante  disponibles= 10526  películas=84
Animation  antes_2010        disponibles= 11303  películas=58
Comedy     2010_en_adelante  disponibles= 24629  películas=345
Comedy     antes_2010        disponibles= 47177  películas=282
Drama      2010_en_adelante  disponibles= 37634  películas=273
Drama      antes_2010        disponibles= 46111  películas=206
Horror     2010_en_adelante  disponibles=  7859  películas=46
Horror     antes_2010        disponibles=  9001  películas=34

Total de la muestra: 6400

Género x periodo:
periodo    2010_en_adelante  antes_2010
genero                                 
Action                  640         640
Animation               640         640
Comedy                  640         640
Drama                   640         640
Horror                  640         640

Paquete x género:
genero   Action  Animation  C

### Guardar los 3 paquetes (y comprobar la muestra)

Si los paquetes ya existen en la carpeta (porque ya se tradujeron), se comprueba que la muestra de hoy sea idéntica a la guardada, para no mezclar reseñas con traducciones de otra muestra. Con la misma semilla debe salir igual.

In [4]:
rutas = [f"{CARPETA}/paquete_{i}_en.csv" for i in range(1, N_PAQUETES + 1)]

if all(os.path.exists(r) for r in rutas):
    guardada = pd.concat([pd.read_csv(r) for r in rutas])
    igual = set(zip(guardada["id"], guardada["texto_en"])) == set(zip(m["id"], m["texto_en"]))
    print("La muestra coincide con la ya guardada:", igual)
    if not igual:
        print("AVISO: se usa la muestra guardada para no desalinear las traducciones.")
        m = guardada.sort_values("id").reset_index(drop=True)
else:
    for i in range(1, N_PAQUETES + 1):
        m[m["paquete"] == i].to_csv(f"{CARPETA}/paquete_{i}_en.csv", index=False, encoding="utf-8-sig")
    print("Paquetes guardados en", CARPETA)

print(m["paquete"].value_counts().sort_index())

La muestra coincide con la ya guardada: True
paquete
1    2140
2    2130
3    2130
Name: count, dtype: int64


## 3. Traducción automática (MarianMT)

Se usa Helsinki-NLP/opus-mt-en-es, que es gratuito y corre de forma local. Como el modelo trunca las entradas largas, cada reseña se divide en oraciones, se traducen por lotes y se vuelven a unir.

La traducción guarda su avance en un checkpoint cada 100 reseñas: si Colab se desconecta, al ejecutar de nuevo continúa donde quedó. Si ya está todo traducido, esta celda solo lo confirma.

> Las traducciones no se corrigen: se analizan tal como las produce el modelo. Solo se eliminan después las que fallaron por completo o quedaron cortadas.

Limitación conocida: el modelo lee como máximo 256 tokens por oración y descarta el resto. Muchas reseñas en inglés traen el punto pegado a la oración siguiente (film.It was...), el divisor de oraciones no corta ahí y queda una oración larguísima cuya parte final se pierde en la traducción. Esas reseñas se detectan y se eliminan en la limpieza (sección 4).

Las funciones de traducción y limpieza están definidas en la siguiente celda

In [5]:
#Traducción
def traducir_oraciones(oraciones, tok, mt, device, lote=64):
    """Traduce una lista de oraciones por lotes. Las ordena por largo para que
    cada lote tenga poco relleno (es más rápido) y devuelve el resultado en el orden original."""
    import torch
    orden = sorted(range(len(oraciones)), key=lambda i: len(oraciones[i]))
    salida = [None] * len(oraciones)
    for k in range(0, len(orden), lote):
        indices = orden[k:k + lote]
        entrada = tok([oraciones[i] for i in indices], return_tensors="pt",
                      padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generado = mt.generate(**entrada)
        textos = tok.batch_decode(generado, skip_special_tokens=True)
        for i, t in zip(indices, textos):
            salida[i] = t
    return salida


def traducir_textos(textos, tok, mt, device):
    """Traduce reseñas completas: las divide en oraciones, traduce todas juntas
    y vuelve a unirlas (el modelo trunca las entradas largas)."""
    from nltk.tokenize import sent_tokenize
    por_resena = [sent_tokenize(t) for t in textos]
    todas = [s for oraciones in por_resena for s in oraciones]
    traducidas = traducir_oraciones(todas, tok, mt, device)
    resultado, pos = [], 0
    for oraciones in por_resena:
        resultado.append(" ".join(traducidas[pos:pos + len(oraciones)]))
        pos += len(oraciones)
    return resultado


#Limpieza
def limpiar_texto(texto):
    """Quita ruido del texto SIN pasar a minúsculas ni quitar tildes ni puntuación normal
    (spaCy los necesita para etiquetar bien)."""
    import re
    if not isinstance(texto, str):
        return ""
    texto = texto.replace("\u2044", "/")                      #1⁄2 -> 1/2
    texto = re.sub(r"<[^>]+>", " ", texto)                    #etiquetas HTML
    texto = re.sub(r"[-_=~*#•·]{4,}", " ", texto)             #líneas decorativas: *****, -----
    texto = re.sub(r"([!?¡¿.,;:])\1{3,}", r"\1\1\1", texto)   #!!!!!!! -> !!!
    texto = re.sub(r"(?<!\S)[¡¿]+(?=\s|$)", " ", texto)       #"¡¡¡" suelto: error del traductor
    texto = re.sub(r"[¡¿]+(?=[!?])", "", texto)               #¡¡¡!!! -> !!!
    texto = re.sub(r"¡{2,}(?=\w)", "¡", texto)                #¡¡Peter -> ¡Peter
    texto = re.sub(r"(?:\s[^\w\s]+){4,}", " ", texto)         #"* * * *": símbolos sueltos
    texto = re.sub(r"\s+", " ", texto).strip()                #espacios repetidos
    texto = re.sub(r"([a-záéíóúñ][.!?])(?=[A-ZÁÉÍÓÚ¿¡])",
                   r"\1 ", texto)                             #corregir espacios
    return texto


PALABRAS_INGLES = ["the", "and", "is", "was", "it", "of", "to", "in", "that", "this",
                   "with", "for", "but", "not", "are", "have", "had", "you", "his", "her"]


def proporcion_ingles(texto):
    """Proporción de palabras del texto que son palabras muy comunes del inglés."""
    palabras = texto.lower().split()
    if len(palabras) == 0:
        return 0
    cuantas = 0
    for p in palabras:
        if p in PALABRAS_INGLES:
            cuantas += 1
    return cuantas / len(palabras)


LIMITE_TOKENS = 240    #el modelo lee como máximo 256 tokens por oración; con 240 queda margen


def tokens_aproximados(oracion):
    """Estima cuántos tokens ocupa una oración para el modelo de traducción
    (cada signo de puntuación cuenta 1 y cada palabra, uno por cada ~4,5 letras)."""
    import re, math
    total = 0
    for pieza in re.findall(r"\w+|[^\w\s]", oracion):
        if pieza[0].isalnum() and pieza[0] != "_":
            total += max(1, math.ceil(len(pieza) / 4.5))
        else:
            total += 1
    return total


def oracion_mas_larga(texto):
    """Tokens aproximados de la oración más larga de una reseña."""
    from nltk.tokenize import sent_tokenize
    return max(tokens_aproximados(o) for o in sent_tokenize(texto))


def motivo_descarte(fila):
    """Devuelve por qué se descarta una reseña, o "" si está bien."""
    import re
    es = fila["texto_es"]
    en = fila["texto_en"]
    n_es = len(es.split())
    n_en = len(en.split())
    if es == "":
        return "vacia"
    if es == en:
        return "sin_traducir"
    if proporcion_ingles(es) > 0.15:
        return "queda_en_ingles"
    if "Ã" in es or "Â" in es:
        return "codificacion_rota"
    if oracion_mas_larga(en) > LIMITE_TOKENS:
        #El modelo trunca lo que pase de 256 tokens por oración: la traducción quedó cortada
        return "traduccion_cortada"
    if n_en > 0 and (n_es / n_en < 0.7 or n_es / n_en > 3):
        return "traduccion_incompleta"
    if re.search(r"\b(\w+)\b(?:[ ,.]+\1\b){5,}", es):
        return "palabra_repetida"
    return ""

In [6]:
MODELO = "Helsinki-NLP/opus-mt-en-es"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if device == "cpu":
    print("AVISO: sin GPU la traducción tarda horas. Activa Entorno de ejecución > Cambiar tipo > T4 GPU.")

tok = MarianTokenizer.from_pretrained(MODELO)
mt = MarianMTModel.from_pretrained(MODELO).to(device)
if device == "cuda":
    mt = mt.half()      #más rápido y con menos memoria

TAM_CHECKPOINT = 100

def traducir_paquetes():
    for paquete in range(1, N_PAQUETES + 1):
        ruta_checkpoint = f"{CARPETA}/ckpt_paquete_{paquete}.csv"
        df_paquete = m[m["paquete"] == paquete]

        ya_hechas = set()
        if os.path.exists(ruta_checkpoint):
            ya_hechas = set(pd.read_csv(ruta_checkpoint)["id"])
        pendientes = df_paquete[~df_paquete["id"].isin(ya_hechas)]
        print(f"\nPaquete {paquete}: {len(df_paquete)} reseñas | ya traducidas: {len(ya_hechas)} | pendientes: {len(pendientes)}")

        inicio = time.time()
        for i in range(0, len(pendientes), TAM_CHECKPOINT):
            lote = pendientes.iloc[i:i + TAM_CHECKPOINT]
            salida = pd.DataFrame({"id": lote["id"].values,
                                   "texto_es": traducir_textos(lote["texto_en"].tolist(), tok, mt, device)})
            salida.to_csv(ruta_checkpoint, mode="a", header=not os.path.exists(ruta_checkpoint), index=False)
            print(f"  {min(i + TAM_CHECKPOINT, len(pendientes))}/{len(pendientes)} | {(time.time() - inicio) / 60:.1f} min")

        # Unir la traducción con los datos originales del paquete
        traducciones = pd.read_csv(ruta_checkpoint).drop_duplicates("id")
        final = df_paquete.merge(traducciones, on="id", how="left")
        final.to_csv(f"{CARPETA}/paquete_{paquete}_es.csv", index=False, encoding="utf-8-sig")
        print(f"  Sin traducir: {final['texto_es'].isna().sum()} | texto igual al inglés: {(final['texto_es'] == final['texto_en']).sum()}")

traducir_paquetes()

Dispositivo: cuda


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Paquete 1: 2140 reseñas | ya traducidas: 2140 | pendientes: 0
  Sin traducir: 0 | texto igual al inglés: 0

Paquete 2: 2130 reseñas | ya traducidas: 2130 | pendientes: 0
  Sin traducir: 0 | texto igual al inglés: 0

Paquete 3: 2130 reseñas | ya traducidas: 2130 | pendientes: 0
  Sin traducir: 0 | texto igual al inglés: 0


### Ejemplos de traducción

Se muestran algunas reseñas en inglés y en español para revisar la calidad.

In [7]:
muestra_ej = pd.read_csv(f"{CARPETA}/paquete_1_es.csv").sample(4, random_state=1)
for _, fila in muestra_ej.iterrows():
    print("EN:", fila["texto_en"][:300])
    print("ES:", fila["texto_es"][:300])
    print()

EN: Two very unexpected things happened to me while viewing "Up." First, I cried. I actually found myself misty-eyed over an opening set piece in which our hero meets, marries and loses his wife to old age. There is no dialogue in this sequence. The entire story is told in images and music. I had no exp
ES: Me pasaron dos cosas muy inesperadas mientras veía "Arriba". Primero, lloré. En realidad me encontré con los ojos borrosos por una pieza de apertura en la que nuestro héroe se encuentra, se casa y pierde a su esposa en la vejez. No hay diálogo en esta secuencia. Toda la historia se cuenta en imágene

EN: Now, I am a Lynch fan. I love the way he is able to tap into the most disturbing recesses of the subconscious, dredge up the most visceral, unnerving and insane thoughts, ideas and images, which reside there, and transfer them onto the screen. This was done superbly in Eraserhead and the pilot of Tw
ES: Ahora, soy un fan de Lynch. Me encanta la forma en que es capaz de aprovechar lo

## 4. Unión de los paquetes y limpieza

Unión: se juntan los 3 paquetes traducidos en resenas_es.csv, que conserva el texto en inglés junto al español.

Limpieza (reglas propias, el PDF solo pide eliminar HTML, vacías, duplicadas y traducciones fallidas):

- Se quitan etiquetas HTML, líneas decorativas (*****), signos repetidos (!!!!!! pasa a !!!), símbolos sueltos que genera el traductor y espacios repetidos.
- No se pasa a minúsculas, no se quitan tildes y no se quita la puntuación normal, porque spaCy los necesita.
- Se descartan reseñas vacías, sin traducir, que quedaron en inglés, con codificación rota, con una palabra repetida muchas veces, duplicadas y con traducción cortada.
- Traducción cortada: se estima cuántos tokens ocupa la oración más larga del original; si pasa de 240 (el modelo corta en 256), la parte final se perdió y la reseña se elimina (traduccion_cortada). Es un criterio aproximado y conservador: se probó con reseñas revisadas a mano y detectó todas las cortadas, aunque puede eliminar algunas completas.
- También se descarta una traducción con menos del 70 % de las palabras del original (traduccion_incompleta).
- Cada descarte se guarda en descartadas.csv con su motivo, para poder revisarlo.

In [8]:
paquetes_es = [pd.read_csv(f"{CARPETA}/paquete_{i}_es.csv") for i in range(1, N_PAQUETES + 1)]
df = pd.concat(paquetes_es).sort_values("id").reset_index(drop=True)
print("Total unido:", len(df))
df.to_csv(f"{CARPETA}/resenas_es.csv", index=False, encoding="utf-8-sig")

Total unido: 6400


In [9]:
df["texto_es"] = df["texto_es"].apply(limpiar_texto)
df["texto_en"] = df["texto_en"].apply(limpiar_texto)

df["motivo"] = df.apply(motivo_descarte, axis=1)
df.loc[(df["motivo"] == "") & df.duplicated("texto_es"), "motivo"] = "duplicada"

print("Reseñas descartadas por motivo:")
print(df[df["motivo"] != ""]["motivo"].value_counts())

Reseñas descartadas por motivo:
motivo
traduccion_cortada       29
traduccion_incompleta     7
palabra_repetida          2
codificacion_rota         1
Name: count, dtype: int64


In [10]:
descartadas = df[df["motivo"] != ""]
limpias = df[df["motivo"] == ""].drop(columns=["motivo", "paquete"])

descartadas.to_csv(f"{CARPETA}/descartadas.csv", index=False, encoding="utf-8-sig")
limpias.to_csv(f"{CARPETA}/resenas_limpias.csv", index=False, encoding="utf-8-sig")
print("Reseñas limpias guardadas:", len(limpias))
descartadas[["id", "motivo", "texto_es"]].head(10)

Reseñas limpias guardadas: 6361


,id,motivo,texto_es
193,194,traduccion_incompleta,"Lo que ""le roi et l'oiseau"" (Jacques Prévert y..."
283,284,codificacion_rota,"Anoche atrapé la mayor parte de esto en HD, pe..."
525,526,traduccion_cortada,Esto es lo que he visto 10 películas de este t...
675,676,traduccion_cortada,"Bueno, tengo que ver este divertido viaje Adam..."
677,678,traduccion_cortada,Esto es todo lo que necesito decir...El puente...
726,727,traduccion_cortada,Yo estaba en la casa de alguien que tenía esta...
778,779,traduccion_cortada,Ya hace bastantes años que se puede decir - es...
898,899,palabra_repetida,Vi una película en los cines y fue una de las ...
1301,1302,traduccion_cortada,Probablemente una de las peores películas que ...
1379,1380,traduccion_cortada,¿Cómo le dieron un siete esto es como una de l...


## 5. Verificación del corpus final

Se comprueba que el corpus cumple los requisitos del PDF (sección 4.3): al menos 3.000 reseñas, 5 géneros con 400 o más reseñas cada uno, 2 periodos con 1.000 o más reseñas cada uno y longitud entre 30 y 400 palabras.

In [11]:
print("Total de reseñas limpias:", len(limpias), "(mínimo 3.000)")
print("\nGénero x periodo (mínimo 400 por género y 1.000 por periodo):")
tabla = pd.crosstab(limpias["genero"], limpias["periodo"], margins=True)
print(tabla)

print("\nValoración por género:")
print(pd.crosstab(limpias["genero"], limpias["valoracion"]))

palabras_en = limpias["texto_en"].str.split().str.len()
print("\nPalabras por reseña (texto original en inglés):")
print(palabras_en.describe().round(1))
print("\nLongitud media en palabras del texto en español, por periodo:")
print(limpias.groupby("periodo")["texto_es"].apply(lambda s: s.str.split().str.len().mean()).round(1))

assert len(limpias) >= 3000
assert (tabla.loc[GENEROS, "All"] >= 400).all()
assert (tabla.loc["All", ["antes_2010", "2010_en_adelante"]] >= 1000).all()
print("\nRequisitos del corpus: OK")

Total de reseñas limpias: 6361 (mínimo 3.000)

Género x periodo (mínimo 400 por género y 1.000 por periodo):
periodo    2010_en_adelante  antes_2010   All
genero                                       
Action                  636         636  1272
Animation               635         639  1274
Comedy                  633         636  1269
Drama                   637         638  1275
Horror                  637         634  1271
All                    3178        3183  6361

Valoración por género:
valoracion  intermedia  negativa  positiva
genero                                    
Action             201       301       770
Animation          156       162       956
Comedy             184       227       858
Drama              158       243       874
Horror             219       280       772

Palabras por reseña (texto original en inglés):
count    6361.0
mean      185.7
std        83.8
min        30.0
25%       126.0
50%       167.0
75%       237.0
max       405.0
Name: texto_en, dtype

In [12]:
print("Archivos en", CARPETA)
for nombre in sorted(os.listdir(CARPETA)):
    if nombre.endswith(".csv"):
        mb = os.path.getsize(f"{CARPETA}/{nombre}") / 1_000_000
        print(f"  {nombre:28s} {mb:6.1f} MB")

Archivos en /content/drive/MyDrive/proyecto_pos
  ckpt_paquete_1.csv              2.5 MB
  ckpt_paquete_2.csv              2.5 MB
  ckpt_paquete_3.csv              2.4 MB
  descartadas.csv                 0.1 MB
  muestra_completa_en.csv         7.1 MB
  paquete_1_en.csv                2.4 MB
  paquete_1_es.csv                4.8 MB
  paquete_2_en.csv                2.4 MB
  paquete_2_es.csv                4.8 MB
  paquete_3_en.csv                2.4 MB
  paquete_3_es.csv                4.8 MB
  resenas_es.csv                 14.5 MB
  resenas_limpias.csv            14.3 MB


In [13]:
#Para verificar la cantidad de reseñas
import pandas as pd
d = pd.read_csv("/content/drive/MyDrive/proyecto_pos/resenas_limpias.csv")
print(d.shape)
print("paquete" in d.columns)

(6361, 11)
False


## Archivos generados

Archivo

resenas_es.csv,
resenas_limpias.csv ,
descartadas.csv

Contenido

Los 3 paquetes antes y después de traducir,
Muestra traducida completa (6.400 reseñas), sin limpiar,
Corpus final para el etiquetado POS
 Reseñas eliminadas y su motivo

Columnas de resenas_limpias.csv: id, movie_id, genero, periodo, anio_review, fecha, calificacion, valoracion, is_spoiler, texto_en, texto_es.